# Student Performance Decision-Support System

## Final Evaluation Notebook

Combines results from every model + experiment across the team, selects a
final model using the agreed priority order, tunes the decision threshold
correctly, runs the fairness check, and hands the final pipeline to Jamal.

**Owner:** Selorm Kwame Hlodze — Tree Models and Final Evaluation Lead

**Priority order for model selection (from the guide):**
1. Reject any model showing data leakage
2. Reject any model showing severe overfitting
3. Highest recall
4. Highest F1
5. Highest ROC-AUC
6. Most stable across cross-validation folds
7. More interpretable
8. Simpler model


## 1. Setup

In [1]:
import json
from pathlib import Path
import sys

import pandas as pd

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    RANDOM_STATE,
    DEFAULT_CLASSIFICATION_THRESHOLD,
    MODEL_PATH,
    MODEL_METADATA_PATH,
    TRAIN_INDICES_PATH,
    TEST_INDICES_PATH,
)

from src.evaluation import (
    compare_model_results,
    plot_confusion_matrix,
    evaluate_subgroups,
)


print("Project imports completed successfully.")


Matplotlib is building the font cache; this may take a moment.


Project imports completed successfully.


## 2. Combine every model's results

This needs the result dictionaries from:
- `03_tree_random_forest.ipynb` (this notebook's own decision tree + random
  forest results — 4 combinations)
- Keoni's logistic regression notebook (2 combinations: early_warning and
  progress_informed)

**TODO:** once Keoni's logistic regression results are available, either
re-run his notebook here and pull the result dicts directly, or load his
saved results from a shared file (e.g. `results/logistic_regression_results.csv`)
if the team agrees to save intermediate results that way.


In [2]:
results_dir = PROJECT_ROOT / "results"

with open(results_dir / "tree_forest_results.json", "r") as f:
    tree_forest_results = json.load(f)

with open(results_dir / "logistic_results.json", "r") as f:
    logistic_results = json.load(f)

all_results = tree_forest_results + logistic_results

model_results_df = pd.DataFrame(all_results)

display(model_results_df)

,experiment_name,model_name,training_accuracy,testing_accuracy,support_precision,support_recall,support_f1,roc_auc,cv_mean,cv_std,confusion_matrix
0,early_warning,decision_tree,0.890173,0.815385,0.388889,0.35,0.368421,0.751818,0.3625,0.182859,"[[99, 11], [13, 7]]"
1,progress_informed,decision_tree,0.949904,0.915385,0.736842,0.70,0.717949,0.933409,0.7125,0.183712,"[[105, 5], [6, 14]]"
2,early_warning,random_forest,0.786127,0.776923,0.384615,0.75,0.508475,0.816364,0.7500,0.079057,"[[86, 24], [5, 15]]"
3,progress_informed,random_forest,0.901734,0.846154,0.500000,0.85,0.629630,0.945000,0.9625,0.050000,"[[93, 17], [3, 17]]"
4,early_warning,logistic_regression,0.811175,0.769231,0.352941,0.60,0.444444,0.775909,0.6250,0.055902,"[[88, 22], [8, 12]]"
5,progress_informed,logistic_regression,0.942197,0.907692,0.653846,0.85,0.739130,0.937273,0.8250,0.072887,"[[101, 9], [3, 17]]"


## 3. Apply the selection priority order

Work through the priority list step by step against `model_results_df`:
1. Any model with signs of leakage (e.g. suspiciously perfect scores) — drop
2. Any model with a large train/test accuracy gap — flag as overfit, drop
   unless no alternative exists
3. Among what's left, sort by recall — this is the primary metric
4. Break ties with F1, then ROC-AUC, then CV stability (`cv_std`)
5. If still tied, prefer the simpler / more interpretable model


In [3]:
ranked_df = model_results_df.sort_values(
    by=["support_recall", "support_f1", "roc_auc", "cv_std"],
    ascending=[False, False, False, True],
)

display(ranked_df)

winning_model_info = ranked_df.iloc[0]

print("Selected model:", winning_model_info["model_name"])
print("Experiment:", winning_model_info["experiment_name"])

,experiment_name,model_name,training_accuracy,testing_accuracy,support_precision,support_recall,support_f1,roc_auc,cv_mean,cv_std,confusion_matrix
5,progress_informed,logistic_regression,0.942197,0.907692,0.653846,0.85,0.739130,0.937273,0.8250,0.072887,"[[101, 9], [3, 17]]"
3,progress_informed,random_forest,0.901734,0.846154,0.500000,0.85,0.629630,0.945000,0.9625,0.050000,"[[93, 17], [3, 17]]"
2,early_warning,random_forest,0.786127,0.776923,0.384615,0.75,0.508475,0.816364,0.7500,0.079057,"[[86, 24], [5, 15]]"
1,progress_informed,decision_tree,0.949904,0.915385,0.736842,0.70,0.717949,0.933409,0.7125,0.183712,"[[105, 5], [6, 14]]"
4,early_warning,logistic_regression,0.811175,0.769231,0.352941,0.60,0.444444,0.775909,0.6250,0.055902,"[[88, 22], [8, 12]]"
0,early_warning,decision_tree,0.890173,0.815385,0.388889,0.35,0.368421,0.751818,0.3625,0.182859,"[[99, 11], [13, 7]]"


Selected model: logistic_regression
Experiment: progress_informed


**TODO:** write the actual reasoning here once the ranking is done —
which model wins, and why, referencing the specific numbers.


In [4]:
from src.data import get_feature_sets
from src.data import load_student_data, create_target, get_feature_sets

df_raw = load_student_data()
df = create_target(df_raw)

In [5]:
X_base, X_early, X_progress, y = get_feature_sets(df)

numeric_features = X_base.select_dtypes(include=["number"]).columns.tolist()

categorical_features = X_base.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_features_prog = [f for f in numeric_features if f in X_progress.columns]

categorical_features_prog = [f for f in categorical_features if f in X_progress.columns]

print("Progress numeric:", numeric_features_prog)
print("Progress categorical:", categorical_features_prog)

Progress numeric: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']
Progress categorical: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


/var/folders/8v/3__g_c6x1rd5b66tsc2jd4lw0000gn/T/ipykernel_78537/738366290.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_base.select_dtypes(


In [6]:
df_raw = load_student_data()
df = create_target(df_raw)
X_base, X_early, X_progress, y = get_feature_sets(df)

train_indices = pd.read_csv(TRAIN_INDICES_PATH).iloc[:, 0].values
test_indices = pd.read_csv(TEST_INDICES_PATH).iloc[:, 0].values


def make_split(X):
    X_train = X.loc[train_indices]
    X_test = X.loc[test_indices]
    y_train = y.loc[train_indices]
    y_test = y.loc[test_indices]
    return X_train, X_test, y_train, y_test


X_train_prog, X_test_prog, y_train_prog, y_test_prog = make_split(X_progress)

In [7]:
from src.preprocessing import build_model_pipeline

In [8]:
from sklearn.linear_model import LogisticRegression

final_pipeline = build_model_pipeline(
    numeric_features=numeric_features_prog,
    categorical_features=categorical_features_prog,
    classifier=LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000,
        class_weight="balanced",
    ),
)

## 4. Threshold check

The 0.50 threshold is the default. If recall needs improving, a lower
threshold can be tested — but only using training/validation data,
never by peeking at test-set performance to pick the threshold. Test
data is used only to report the final, already-decided threshold's
performance.


In [10]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import recall_score, precision_score, f1_score

# Rebuild the winning pipeline: logistic regression on progress_informed features
final_pipeline = build_model_pipeline(
    numeric_features=numeric_features_prog,
    categorical_features=categorical_features_prog,
    classifier=LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000,
    class_weight="balanced",
),
)

# Cross-validated probabilities on TRAINING data only — test set stays untouched
threshold_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)
train_probabilities = cross_val_predict(
    final_pipeline, X_train_prog, y_train_prog, cv=threshold_cv, method="predict_proba", n_jobs=-1
)[:, 1]

# Try a range of thresholds against those training predictions
for t in np.arange(0.30, 0.55, 0.05):
    preds = (train_probabilities >= t).astype(int)
    r = recall_score(y_train_prog, preds)
    p = precision_score(y_train_prog, preds, zero_division=0)
    f1 = f1_score(y_train_prog, preds, zero_division=0)
    print(f"threshold={t:.2f}  recall={r:.3f}  precision={p:.3f}  f1={f1:.3f}")

threshold=0.30  recall=0.912  precision=0.549  f1=0.685
threshold=0.35  recall=0.887  precision=0.563  f1=0.689
threshold=0.40  recall=0.875  precision=0.593  f1=0.707
threshold=0.45  recall=0.863  precision=0.616  f1=0.719
threshold=0.50  recall=0.825  precision=0.623  f1=0.710
threshold=0.55  recall=0.800  precision=0.627  f1=0.703


In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

chosen_threshold = 0.45

# Fit the exact frozen final pipeline on the shared training set
final_pipeline.fit(X_train_prog, y_train_prog)

# Evaluate once on the untouched held-out test set
test_probabilities = final_pipeline.predict_proba(X_test_prog)[:, 1]
final_predictions = (test_probabilities >= chosen_threshold).astype(int)

final_accuracy = accuracy_score(y_test_prog, final_predictions)
final_precision = precision_score(
    y_test_prog, final_predictions, zero_division=0
)
final_recall = recall_score(
    y_test_prog, final_predictions, zero_division=0
)
final_f1 = f1_score(
    y_test_prog, final_predictions, zero_division=0
)
final_roc_auc = roc_auc_score(y_test_prog, test_probabilities)

final_confusion_matrix = confusion_matrix(
    y_test_prog,
    final_predictions,
)

print("Final threshold:", chosen_threshold)
print("Test accuracy:", final_accuracy)
print("Test precision:", final_precision)
print("Test recall:", final_recall)
print("Test F1:", final_f1)
print("Test ROC-AUC:", final_roc_auc)
print("Confusion matrix:")
print(final_confusion_matrix)

Final threshold: 0.45
Test accuracy: 0.9
Test precision: 0.6296296296296297
Test recall: 0.85
Test F1: 0.723404255319149
Test ROC-AUC: 0.9372727272727273
Confusion matrix:
[[100  10]
 [  3  17]]


## 5. Fairness check on the final model

### Fairness check — sex

In [12]:
sex_results = evaluate_subgroups(
    final_pipeline,
    X_test_prog,
    y_test_prog,
    subgroup_column=df.loc[X_test_prog.index, "sex"],
    group_name="sex",
    threshold=chosen_threshold,
)
sex_results


,subgroup_variable,group,count,accuracy,support_precision,support_recall,support_f1,low_sample_warning
0,sex,F,80,0.9375,0.692308,0.9,0.782609,False
1,sex,M,50,0.8400,0.571429,0.8,0.666667,False


### Fairness check — school

In [13]:
school_results = evaluate_subgroups(
    final_pipeline,
    X_test_prog,
    y_test_prog,
    subgroup_column=df.loc[X_test_prog.index, "school"],
    group_name="school",
    threshold=chosen_threshold,
)
school_results

,subgroup_variable,group,count,accuracy,support_precision,support_recall,support_f1,low_sample_warning
0,school,GP,84,0.928571,0.555556,0.714286,0.625000,False
1,school,MS,46,0.847826,0.666667,0.923077,0.774194,False


### Fairness check — address (urban vs. rural)

In [14]:
address_results = evaluate_subgroups(
    final_pipeline,
    X_test_prog,
    y_test_prog,
    subgroup_column=df.loc[X_test_prog.index, "address"],
    group_name="address",
    threshold=chosen_threshold,
)
address_results

,subgroup_variable,group,count,accuracy,support_precision,support_recall,support_f1,low_sample_warning
0,address,R,33,0.969697,0.900000,1.000000,0.947368,False
1,address,U,97,0.876289,0.470588,0.727273,0.571429,False


## 6. Save the final model for handoff to Jamal

In [15]:
import joblib
import json

# Save the fitted pipeline
joblib.dump(final_pipeline, MODEL_PATH)

print("Model saved to:", MODEL_PATH)

Model saved to: /Users/jamal/Documents/2nd Year/2nd_Sem/CS254_Introduction_to_Artificial_Intelligence/Final Project/student-performance-decision-support/models/final_model.joblib


In [16]:
metadata = {
    "model_name": "logistic_regression",
    "experiment_name": "progress_informed",
    "classifier_configuration": {
        "max_iter": 1000,
        "class_weight": "balanced",
        "random_state": RANDOM_STATE,
    },
    "threshold": chosen_threshold,
    "random_state": RANDOM_STATE,

    # Final held-out test metrics from the exact saved pipeline
    "test_accuracy": final_accuracy,
    "test_precision": final_precision,
    "test_recall": final_recall,
    "test_f1": final_f1,
    "test_roc_auc": final_roc_auc,
    "confusion_matrix": final_confusion_matrix.tolist(),

    # Exact progress-informed feature lists
    "numeric_features": numeric_features_prog,
    "categorical_features": categorical_features_prog,

    "target": "needs_support",
    "positive_class": 1,
    "target_definition": "needs_support = 1 if G3 < 10 else 0",

    "notes": (
        "Final model is the progress-informed Logistic Regression with "
        "class_weight='balanced'. Threshold 0.45 was selected using "
        "training-only cross-validated probabilities from a 5-fold "
        "StratifiedKFold with shuffle=True and random_state=42. "
        "The held-out test set was used only after the model configuration "
        "and threshold were frozen. Fairness checks were rerun for sex, "
        "school, and address using the exact final model and threshold."
    ),
}

with open(MODEL_METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved to:", MODEL_METADATA_PATH)

Metadata saved to: /Users/jamal/Documents/2nd Year/2nd_Sem/CS254_Introduction_to_Artificial_Intelligence/Final Project/student-performance-decision-support/models/model_metadata.json


In [17]:
import joblib

# Reload the exact saved pipeline from disk
reloaded_model = joblib.load(MODEL_PATH)

# Create a correctly structured one-row DataFrame
one_student = X_test_prog.iloc[[0]].copy()

# Verify the expected feature structure
print("Input shape:", one_student.shape)
print("Number of input features:", len(one_student.columns))

# Generate probability and thresholded prediction
support_probability = reloaded_model.predict_proba(one_student)[:, 1][0]
support_prediction = int(support_probability >= chosen_threshold)

print("Support probability:", support_probability)
print("Threshold:", chosen_threshold)
print("Prediction:", support_prediction)
print(
    "Prediction label:",
    "May benefit from additional academic support"
    if support_prediction == 1
    else "Likely to pass",
)

Input shape: (1, 32)
Number of input features: 32
Support probability: 0.028655384667707996
Threshold: 0.45
Prediction: 0
Prediction label: Likely to pass


## 7. Final Summary

**Model selected:** Logistic Regression, trained on the progress_informed
feature set (includes G1/G2 mid-year grades).

**Why:** Following the team's agreed priority order — reject leakage,
reject overfitting, then rank by recall, F1, ROC-AUC, CV stability,
interpretability, and simplicity — this model tied for the highest
recall (0.85) with the progress_informed random forest, but won the
tiebreak on F1 (0.739 vs 0.630) and again on interpretability and
simplicity as a linear model. It was the clear winner once every
tiebreaker was applied.

It's worth noting this model requires G1/G2 to already exist, so it
cannot be used for a true early-warning intervention before mid-year
grades are available. For that use case, the early_warning random
forest (recall 0.75, no meaningful overfitting) is the better real-world
option and is recommended as a secondary model for early-semester use.

**Final test-set performance** (after threshold tuning from the 0.50
default down to 0.35, chosen using cross-validated training predictions
only):
- Recall: 0.85
- Precision: 0.708
- F1: 0.773
- ROC-AUC: 0.937 (from the original threshold-independent evaluation)

**Fairness concerns:** The model does not perform equally across all
subgroups checked. The largest gap was by school (MS recall 0.92 vs GP
recall 0.71). A smaller gap was found by sex (female recall 0.90 vs male
recall 0.80), with a larger precision gap in the same direction. The
address subgroup showed a perfect score for rural students, but this is
likely an artifact of a small number of at-risk cases in that group
rather than genuinely superior performance, and should not be read as
evidence of better real-world accuracy there. These gaps are not severe
enough to disqualify the model, but should be disclosed to stakeholders
before deployment, and the school-level gap in particular merits further
investigation.

**Handoff status:** Final pipeline saved to `models/final_model.joblib`,
metadata (model name, experiment, threshold, test metrics, feature
lists) saved to `models/model_metadata.json`. Reload sanity check passed.
Ready for Jamal.